# Seattle Weather — Decision Tree vs Random Forest

This notebook builds two **classification models** to predict `RAIN` from the Seattle weather dataset:

1. **Decision Tree Classifier**
2. **Random Forest Classifier**

The workflow intentionally checks:

- missing/null values
- duplicate rows
- invalid date/target values
- class balance before and after train/test splitting
- outliers using the IQR method
- target leakage
- best Decision Tree `max_depth`
- best Random Forest `max_depth`
- best Random Forest `n_estimators`
- final test-set metrics and confusion matrices
- saving both trained models into a single pickle file for a future Streamlit `WP.py`

> **Important:** `PRCP` is not used as a predictive feature. In this dataset, `RAIN` is exactly determined by whether precipitation is greater than zero, so using `PRCP` would be target leakage and would make the model look artificially accurate.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42
DATA_PATH = "seattle_weather_1948-2017.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nData types:")
display(df.dtypes)


Shape: (25551, 5)


,DATE,PRCP,TMAX,TMIN,RAIN
0,1948-01-01,0.47,51,42,True
1,1948-01-02,0.59,45,36,True
2,1948-01-03,0.42,45,35,True
3,1948-01-04,0.31,45,34,True
4,1948-01-05,0.17,45,32,True



Data types:


DATE        str
PRCP    float64
TMAX      int64
TMIN      int64
RAIN     object
dtype: object

## 1. Data quality checks

In [2]:
# Null values
print("Null values per column:")
display(df.isna().sum().to_frame("null_count"))

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Basic descriptive statistics
display(df.describe(include="all"))


Null values per column:


,null_count
DATE,0
PRCP,3
TMAX,0
TMIN,0
RAIN,3


Duplicate rows: 0


,DATE,PRCP,TMAX,TMIN,RAIN
count,25551,25548.000000,25551.000000,25551.000000,25548
unique,25551,NaN,NaN,NaN,2
top,1948-01-01,NaN,NaN,NaN,False
freq,1,NaN,NaN,NaN,14648
mean,NaN,0.106222,59.544206,44.514226,NaN
std,NaN,0.239031,12.772984,8.892836,NaN
min,NaN,0.000000,4.000000,0.000000,NaN
25%,NaN,0.000000,50.000000,38.000000,NaN
50%,NaN,0.000000,58.000000,45.000000,NaN
75%,NaN,0.100000,69.000000,52.000000,NaN


In [3]:
# Parse date and normalize the target
data = df.copy()

data["DATE"] = pd.to_datetime(data["DATE"], errors="coerce")
data["RAIN"] = data["RAIN"].map({
    True: 1, False: 0,
    "True": 1, "False": 0,
    "true": 1, "false": 0
})

print("Invalid/missing DATE:", data["DATE"].isna().sum())
print("Invalid/missing RAIN:", data["RAIN"].isna().sum())

# We need the target and core weather variables to train.
before = len(data)
data = data.dropna(subset=["DATE", "RAIN", "TMAX", "TMIN"]).copy()
print(f"Rows removed during required-field cleaning: {before - len(data)}")
print("Cleaned shape:", data.shape)


Invalid/missing DATE: 0
Invalid/missing RAIN: 3
Rows removed during required-field cleaning: 3
Cleaned shape: (25548, 5)


## 2. Target leakage check

`PRCP` is precipitation. In this particular dataset, the target `RAIN` is effectively:

- `RAIN = False` when `PRCP == 0`
- `RAIN = True` when `PRCP > 0`

Therefore, `PRCP` would reveal the answer directly. It is excluded from the model. This is an important preprocessing decision, not an omission.

In [4]:
leakage_check = data.dropna(subset=["PRCP"]).copy()
leakage_check["RAIN_FROM_PRCP"] = (leakage_check["PRCP"] > 0).astype(int)

leakage_accuracy = (
    leakage_check["RAIN_FROM_PRCP"] == leakage_check["RAIN"].astype(int)
).mean()

print(f"PRCP -> RAIN direct-rule agreement: {leakage_accuracy:.2%}")
print("PRCP will NOT be used as a model feature.")


PRCP -> RAIN direct-rule agreement: 100.00%
PRCP will NOT be used as a model feature.


## 3. Outlier analysis

In [5]:
# IQR outlier report.
# Weather extremes are legitimate observations, so we REPORT them rather than
# automatically deleting them. Tree-based models do not require scaling.

numeric_for_outliers = ["PRCP", "TMAX", "TMIN"]

outlier_rows = []
for col in numeric_for_outliers:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[col] < lower) | (data[col] > upper)

    outlier_rows.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_percent": float(mask.mean() * 100)
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

print("Decision: no automatic outlier deletion.")
print("Reason: extreme temperatures/precipitation can be valid weather events.")


,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percent
0,PRCP,0.0,0.1,0.1,-0.15,0.25,3558,13.926726
1,TMAX,50.0,69.0,19.0,21.50,97.50,24,0.093941
2,TMIN,38.0,52.0,14.0,17.00,73.00,92,0.360106


Decision: no automatic outlier deletion.
Reason: extreme temperatures/precipitation can be valid weather events.


## 4. Feature engineering

`DATE` itself is not passed directly to the models. Instead, seasonality is represented with cyclical date features.

`PRCP` is intentionally excluded because of target leakage.

Final features:

- `TMAX`
- `TMIN`
- `month_sin`
- `month_cos`
- `doy_sin`
- `doy_cos`

In [6]:
data["month"] = data["DATE"].dt.month
data["doy"] = data["DATE"].dt.dayofyear

data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

data["doy_sin"] = np.sin(2 * np.pi * data["doy"] / 365.25)
data["doy_cos"] = np.cos(2 * np.pi * data["doy"] / 365.25)

FEATURES = [
    "TMAX", "TMIN",
    "month_sin", "month_cos",
    "doy_sin", "doy_cos"
]
TARGET = "RAIN"

X = data[FEATURES]
y = data[TARGET].astype(int)

print("Features:", FEATURES)
print("\nFull dataset class distribution:")
display(y.value_counts().rename(index={0: "No Rain", 1: "Rain"}).to_frame("count"))
display((y.value_counts(normalize=True) * 100).rename(index={0: "No Rain", 1: "Rain"}).to_frame("percent"))


Features: ['TMAX', 'TMIN', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']

Full dataset class distribution:


,count
RAIN,
No Rain,14648
Rain,10900


,percent
RAIN,
No Rain,57.335212
Rain,42.664788


## 5. Stratified train/test split — before balancing

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training distribution BEFORE balancing:")
display(y_train.value_counts().rename(index={0: "No Rain", 1: "Rain"}).to_frame("count"))

print("Test distribution (kept natural; NOT balanced):")
display(y_test.value_counts().rename(index={0: "No Rain", 1: "Rain"}).to_frame("count"))


Training distribution BEFORE balancing:


,count
RAIN,
No Rain,11718
Rain,8720


Test distribution (kept natural; NOT balanced):


,count
RAIN,
No Rain,2930
Rain,2180


## 6. Balance the training data only

The test set must represent the real dataset distribution, so it is **not** balanced.

For training, the majority class is randomly undersampled to exactly match the minority class. This prevents class imbalance without creating synthetic observations.

In [8]:
train_df = pd.concat([X_train, y_train.rename(TARGET)], axis=1)

class_counts = train_df[TARGET].value_counts()
minority_count = class_counts.min()

balanced_parts = []
for cls in sorted(class_counts.index):
    class_part = train_df[train_df[TARGET] == cls].sample(
        n=minority_count,
        random_state=RANDOM_STATE
    )
    balanced_parts.append(class_part)

balanced_train = (
    pd.concat(balanced_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

X_train_balanced = balanced_train[FEATURES]
y_train_balanced = balanced_train[TARGET].astype(int)

print("Training distribution AFTER balancing:")
display(y_train_balanced.value_counts().rename(index={0: "No Rain", 1: "Rain"}).to_frame("count"))

assert y_train_balanced.value_counts().nunique() == 1, "Training data is not balanced."
print("PASS: training classes are exactly balanced.")


Training distribution AFTER balancing:


,count
RAIN,
No Rain,8720
Rain,8720


PASS: training classes are exactly balanced.


## 7. Tune Decision Tree depth

In [9]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

dt_depths = [2, 3, 4, 5, 6, 8, 10, 12, 15, None]
dt_depth_scores = {}

for depth in dt_depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    score = cross_val_score(
        model, X_train_balanced, y_train_balanced,
        cv=cv, scoring="f1", n_jobs=2
    ).mean()
    dt_depth_scores[str(depth)] = score

dt_depth_table = pd.DataFrame({
    "max_depth": [str(x) for x in dt_depths],
    "mean_cv_f1": [dt_depth_scores[str(x)] for x in dt_depths]
}).sort_values("mean_cv_f1", ascending=False)

display(dt_depth_table)

best_dt_depth = max(dt_depths, key=lambda x: dt_depth_scores[str(x)])
print("Best Decision Tree max_depth:", best_dt_depth)


,max_depth,mean_cv_f1
3,5,0.767831
2,4,0.767112
5,8,0.766700
4,6,0.764044
6,10,0.755864
1,3,0.746671
7,12,0.739200
0,2,0.715338
8,15,0.714907
9,None,0.678808


Best Decision Tree max_depth: 5


## 8. Tune Random Forest depth

In [10]:
rf_depths = [3, 5, 8, 12, None]
rf_depth_scores = {}

for depth in rf_depths:
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=2
    )
    score = cross_val_score(
        model, X_train_balanced, y_train_balanced,
        cv=cv, scoring="f1", n_jobs=2
    ).mean()
    rf_depth_scores[str(depth)] = score

rf_depth_table = pd.DataFrame({
    "max_depth": [str(x) for x in rf_depths],
    "mean_cv_f1": [rf_depth_scores[str(x)] for x in rf_depths]
}).sort_values("mean_cv_f1", ascending=False)

display(rf_depth_table)

best_rf_depth = max(rf_depths, key=lambda x: rf_depth_scores[str(x)])
print("Best Random Forest max_depth:", best_rf_depth)


,max_depth,mean_cv_f1
2,8,0.776400
3,12,0.770882
1,5,0.770584
0,3,0.748640
4,None,0.734006


Best Random Forest max_depth: 8


## 9. Tune Random Forest `n_estimators`

In [11]:
rf_n_estimators = [50, 100, 150, 200]
rf_n_scores = {}

for n in rf_n_estimators:
    model = RandomForestClassifier(
        n_estimators=n,
        max_depth=best_rf_depth,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=2
    )
    score = cross_val_score(
        model, X_train_balanced, y_train_balanced,
        cv=cv, scoring="f1", n_jobs=2
    ).mean()
    rf_n_scores[str(n)] = score

rf_n_table = pd.DataFrame({
    "n_estimators": rf_n_estimators,
    "mean_cv_f1": [rf_n_scores[str(x)] for x in rf_n_estimators]
}).sort_values("mean_cv_f1", ascending=False)

display(rf_n_table)

best_rf_n_estimators = max(
    rf_n_estimators,
    key=lambda x: rf_n_scores[str(x)]
)

print("Best Random Forest n_estimators:", best_rf_n_estimators)


,n_estimators,mean_cv_f1
3,200,0.777401
2,150,0.776466
1,100,0.776400
0,50,0.774515


Best Random Forest n_estimators: 200


## 10. Train final models

In [12]:
decision_tree = DecisionTreeClassifier(
    max_depth=best_dt_depth,
    random_state=RANDOM_STATE
)
decision_tree.fit(X_train_balanced, y_train_balanced)

random_forest = RandomForestClassifier(
    n_estimators=best_rf_n_estimators,
    max_depth=best_rf_depth,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=2
)
random_forest.fit(X_train_balanced, y_train_balanced)

print("Final Decision Tree:", decision_tree)
print("Final Random Forest:", random_forest)


Final Decision Tree: DecisionTreeClassifier(max_depth=5, random_state=42)
Final Random Forest: RandomForestClassifier(max_depth=8, n_estimators=200, n_jobs=2, random_state=42)


## 11. Compare both models on the untouched test set

In [13]:
def evaluate_model(name, model):
    predictions = model.predict(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions)
    }

    print(f"\n{name}")
    print("-" * len(name))
    print(classification_report(
        y_test, predictions,
        target_names=["No Rain", "Rain"],
        zero_division=0
    ))
    print("Confusion matrix:")
    display(pd.DataFrame(
        confusion_matrix(y_test, predictions),
        index=["Actual No Rain", "Actual Rain"],
        columns=["Predicted No Rain", "Predicted Rain"]
    ))

    return metrics

results = pd.DataFrame([
    evaluate_model("Decision Tree", decision_tree),
    evaluate_model("Random Forest", random_forest)
]).set_index("Model")

display(results.round(4))



Decision Tree
-------------
              precision    recall  f1-score   support

     No Rain       0.86      0.66      0.75      2930
        Rain       0.65      0.85      0.74      2180

    accuracy                           0.74      5110
   macro avg       0.76      0.76      0.74      5110
weighted avg       0.77      0.74      0.75      5110

Confusion matrix:


,Predicted No Rain,Predicted Rain
Actual No Rain,1946,984
Actual Rain,320,1860



Random Forest
-------------
              precision    recall  f1-score   support

     No Rain       0.85      0.72      0.78      2930
        Rain       0.69      0.83      0.75      2180

    accuracy                           0.77      5110
   macro avg       0.77      0.78      0.77      5110
weighted avg       0.78      0.77      0.77      5110

Confusion matrix:


,Predicted No Rain,Predicted Rain
Actual No Rain,2106,824
Actual Rain,366,1814


,Accuracy,Precision,Recall,F1,Balanced Accuracy
Model,,,,,
Decision Tree,0.7448,0.6540,0.8532,0.7404,0.7587
Random Forest,0.7671,0.6876,0.8321,0.7530,0.7754


## 12. Save the trained models as a pickle for Streamlit

In [14]:
MODEL_ARTIFACT = {
    "models": {
        "decision_tree": decision_tree,
        "random_forest": random_forest
    },
    "feature_columns": FEATURES,
    "target_column": TARGET,
    "excluded_columns": ["DATE", "PRCP"],
    "preprocessing": (
        "DATE parsed; cyclical month/day-of-year features created; "
        "rows missing DATE/RAIN/TMAX/TMIN removed; PRCP excluded to prevent leakage."
    ),
    "balancing": (
        "Training data only: majority class randomly undersampled to minority class. "
        "Test data kept at natural distribution."
    ),
    "random_state": RANDOM_STATE,
    "best_params": {
        "decision_tree": {
            "max_depth": int(best_dt_depth)
        },
        "random_forest": {
            "max_depth": int(best_rf_depth),
            "n_estimators": int(best_rf_n_estimators)
        }
    },
    "cv_scoring": "F1 score with 3-fold StratifiedKFold",
    "test_metrics": results.to_dict(orient="index")
}

PICKLE_PATH = "WP_weather_models.pkl"
joblib.dump(MODEL_ARTIFACT, PICKLE_PATH)

print(f"Saved: {PICKLE_PATH}")
print(f"File size: {os.path.getsize(PICKLE_PATH) / 1024 / 1024:.2f} MB")

# Verify that the saved artifact can be loaded and used.
loaded = joblib.load(PICKLE_PATH)
sample_predictions = loaded["models"]["random_forest"].predict(X_test.iloc[:5])

print("Pickle load test: PASS")
print("Sample predictions:", sample_predictions.tolist())


Saved: WP_weather_models.pkl
File size: 6.29 MB


/Users/macbookair/Desktop/ML/VE/lib/python3.14/site-packages/joblib/numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


Pickle load test: PASS
Sample predictions: [0, 1, 0, 0, 1]


## 13. Streamlit-ready feature helper

The future `WP.py` can use the same feature engineering. It should collect a date, maximum temperature and minimum temperature, then construct the six features in exactly the same order as training.

`PRCP` is not required by the saved model.

In [15]:
def make_streamlit_features(date_value, tmax, tmin):
    date_value = pd.to_datetime(date_value)

    row = pd.DataFrame([{
        "TMAX": float(tmax),
        "TMIN": float(tmin),
        "month_sin": np.sin(2 * np.pi * date_value.month / 12),
        "month_cos": np.cos(2 * np.pi * date_value.month / 12),
        "doy_sin": np.sin(2 * np.pi * date_value.dayofyear / 365.25),
        "doy_cos": np.cos(2 * np.pi * date_value.dayofyear / 365.25)
    }])

    return row[FEATURES]

# Example:
example_features = make_streamlit_features("2017-09-17", 70, 55)
print(example_features)

print("\nReady for WP.py:")
print("artifact = joblib.load('WP_weather_models.pkl')")
print("dt_prediction = artifact['models']['decision_tree'].predict(features)")
print("rf_prediction = artifact['models']['random_forest'].predict(features)")


   TMAX  TMIN  month_sin     month_cos   doy_sin   doy_cos
0  70.0  55.0       -1.0 -1.836970e-16 -0.971395 -0.237468

Ready for WP.py:
artifact = joblib.load('WP_weather_models.pkl')
dt_prediction = artifact['models']['decision_tree'].predict(features)
rf_prediction = artifact['models']['random_forest'].predict(features)


## Final notes

- Missing values were explicitly checked and required missing rows were removed.
- Duplicate rows were checked.
- Outliers were measured with IQR and retained because weather extremes are legitimate observations.
- The training set was balanced **after** the train/test split.
- The test set was left untouched and keeps the real class distribution.
- `PRCP` was excluded because it directly determines `RAIN` in this dataset.
- Decision Tree and Random Forest were tuned for depth.
- Random Forest was additionally tuned for `n_estimators`.
- Both trained models are stored in `WP_weather_models.pkl`.
- The same feature-engineering logic is included for the future Streamlit app, reducing train/inference mismatch.
